# Real vs Fake Job Post Prediction

Classifies online job postings as real or fake using NLP (TF-IDF) with Decision Tree and Random Forest models.

**Dataset:** Kaggle "Fake Job Postings" (`fake_job_postings.csv`, 17,880 posts)

**Key design choice:** the data is split into train and test FIRST. Class imbalance is handled with `class_weight='balanced'` on the training data only, so the test set is never touched or duplicated. This avoids data leakage.

## 1. Setup

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, classification_report)

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

RANDOM_STATE = 42

## 2. Load the data
Run the next cell, click **Choose files**, and select `fake_job_postings.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()
data = pd.read_csv('fake_job_postings.csv')
print(data.shape)
data.head()

## 3. Explore

In [ ]:
print(data['fraudulent'].value_counts())
print()
print(data['fraudulent'].value_counts(normalize=True).round(4))

plt.figure(figsize=(5,4))
sns.countplot(x='fraudulent', data=data)
plt.title('Class distribution (0 = real, 1 = fake)')
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Combine text fields into one column

In [ ]:
text_cols = ['title', 'location', 'department', 'salary_range', 'company_profile',
             'description', 'requirements', 'benefits', 'employment_type',
             'required_experience', 'required_education', 'industry', 'function']

data[text_cols] = data[text_cols].fillna('')
data['content'] = data[text_cols].agg(' '.join, axis=1)
df = data[['content', 'fraudulent']].copy()
df.head()

## 5. Clean the text

In [ ]:
def decontracted(phrase):
    phrase = re.sub(r"won't", "will not", phrase)
    phrase = re.sub(r"can't", "can not", phrase)
    phrase = re.sub(r"n't", " not", phrase)
    phrase = re.sub(r"'re", " are", phrase)
    phrase = re.sub(r"'s", " is", phrase)
    phrase = re.sub(r"'d", " would", phrase)
    phrase = re.sub(r"'ll", " will", phrase)
    phrase = re.sub(r"'ve", " have", phrase)
    phrase = re.sub(r"'m", " am", phrase)
    return phrase

# keep negations, they can carry meaning
stop_words = set(stopwords.words('english')) - {'no', 'nor', 'not'}

def clean_text(text):
    text = decontracted(text)
    text = re.sub(r'<[^>]+>', ' ', text)          # remove HTML tags
    text = re.sub(r'[^A-Za-z0-9]+', ' ', text)    # keep letters and numbers
    words = [w for w in text.lower().split() if w not in stop_words]
    return ' '.join(words)

df['clean'] = df['content'].apply(clean_text)
df[['clean', 'fraudulent']].head()

## 6. Train / test split (BEFORE any balancing)
70% train, 30% test, stratified so both sets keep the same fake/real ratio.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean'], df['fraudulent'],
    test_size=0.30, stratify=df['fraudulent'], random_state=RANDOM_STATE
)
print('Train:', X_train.shape[0], '| Test:', X_test.shape[0])
print('Fake in train:', y_train.sum(), '| Fake in test:', y_test.sum())

## 7. TF-IDF features
The vectorizer is fitted on the training text only.

In [ ]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=3)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)
print('Train matrix:', X_train_vec.shape)
print('Test matrix: ', X_test_vec.shape)

## 8. Train the models
`class_weight='balanced'` gives more importance to the rare fake posts during training.

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(
        class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        n_jobs=-1, random_state=RANDOM_STATE),
}

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    print(name, 'trained')

## 9. Evaluate
Because fake posts are rare, accuracy alone can look high even for a weak model. Precision, recall and F1 below are for the **fake** class (label 1).

In [ ]:
results = []
probs = {}

for name, model in models.items():
    pred = model.predict(X_test_vec)
    prob = model.predict_proba(X_test_vec)[:, 1]
    probs[name] = prob
    results.append({
        'Model': name,
        'Accuracy':  accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall':    recall_score(y_test, pred),
        'F1-score':  f1_score(y_test, pred),
        'ROC-AUC':   roc_auc_score(y_test, prob),
    })

results_df = pd.DataFrame(results).set_index('Model').round(4)
print(results_df)
results_df.to_csv('model_results.csv')

In [ ]:
for name, model in models.items():
    print('=' * 50)
    print(name)
    print(classification_report(y_test, model.predict(X_test_vec),
                                target_names=['Real', 'Fake']))

## 10. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, model) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, model.predict(X_test_vec))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. ROC curves

In [ ]:
plt.figure(figsize=(6, 5))
for name, prob in probs.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_val:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve')
plt.legend()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Top features (Random Forest)

In [ ]:
rf = models['Random Forest']
feat_names = np.array(tfidf.get_feature_names_out())
top_idx = np.argsort(rf.feature_importances_)[-15:]

plt.figure(figsize=(7, 6))
plt.barh(feat_names[top_idx], rf.feature_importances_[top_idx])
plt.title('Top 15 features: Random Forest')
plt.xlabel('Importance')
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Try it on a new job post

In [ ]:
sample = '''Work from home. Earn 5000 dollars per week. No experience needed.
Send registration fee and bank details to start immediately.'''

vec = tfidf.transform([clean_text(sample)])
for name, model in models.items():
    label = 'FAKE' if model.predict(vec)[0] == 1 else 'REAL'
    print(f'{name}: {label}')

## 14. Download your outputs
This saves the results table and charts to your computer. Upload them to your GitHub repo.

In [ ]:
for f in ['model_results.csv', 'class_distribution.png', 'confusion_matrices.png',
          'roc_curve.png', 'feature_importance.png']:
    files.download(f)